# 02 — Group Study (multivariate Optuna ablation)

11축 동시 탐색 (TPE multivariate). LGBM HP는 default 고정 → 축 효과 + 상호작용만 학습.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*.csv`
- **출력**: `4_output/baseline/group/{optuna.db, trials.csv, param_importance.csv}`
- **참조**: [strategy.md §5](strategy.md), [strategy_common.md §4·§6·§8](../strategy_common.md)

## 1. 환경 설정 + 데이터 로드

In [ ]:
import os, sys

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
    PROJECT_ROOT = '/content/project'
except ImportError:
    %run ../../setup.py
    from utils.config import PROJECT_ROOT

# baseline 격리 모듈/axes 경로 등록
BASELINE_DIR = os.path.join(PROJECT_ROOT, '3_modeling', '0_baseline')
if BASELINE_DIR not in sys.path:
    sys.path.insert(0, BASELINE_DIR)

import warnings
warnings.filterwarnings('ignore')

from utils.data import load_all, get_feat_cols, split_xs
import axes

xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
axes.set_data(xs, xs_dict, ys, feat_cols)

print(f'Feature 수: {len(feat_cols)}')
print(f'Die 수: train={len(xs_dict["train"]):,}, val={len(xs_dict["validation"]):,}, test={len(xs_dict["test"]):,}')

## 2. Optuna study 설정

- Sampler: TPE (seed=None, multivariate=True, group=True) — strategy_common §4
- Pruner: 비활성 (5-fold 다 끝나야 점수 나옴)
- Storage: sqlite (4_output/baseline/group/optuna.db)
- Trial: default 300 (strategy.md §5.4)

In [ ]:
# ── 노트북 상단 단일 변수 (strategy_common §8) ──
N_JOBS = 7         # 14코어 환경 2 노트북 병렬 시
N_TRIALS = 300     # strategy.md §5.4 default
TIMEOUT_SEC = None  # ★ Colab 타임아웃 대비, 초 단위 (None=무제한)
N_ESTIMATORS = 100 # strategy.md §11 default (sklearn LGBM default)

import json
import optuna
from datetime import datetime
from optuna.samplers import TPESampler

OUT_DIR = os.path.join(PROJECT_ROOT, '4_output', 'baseline', 'group')
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, 'optuna.db')
META_JSON = os.path.join(OUT_DIR, 'meta.json')
STORAGE_URL = f'sqlite:///{DB_PATH}'

sampler = TPESampler(
    seed=None,             # 다양성 — strategy_common §7
    multivariate=True,     # 축 결합 분포 학습
    group=True,            # CLF=off 시 conditional skip
)

study = optuna.create_study(
    study_name='baseline_group',
    storage=STORAGE_URL,
    sampler=sampler,
    direction='minimize',
    load_if_exists=True,   # 끊김 후 재실행 시 이어서
)

# meta.json — 재현성용 단일 파일 (run마다 덮어쓰기)
meta = {
    'created':       datetime.now().isoformat(timespec='seconds'),
    'n_jobs':        N_JOBS,
    'n_trials':      N_TRIALS,
    'n_estimators':  N_ESTIMATORS,
    'study_name':    'baseline_group',
    'sampler':       'TPESampler(seed=None, multivariate=True, group=True)',
    'pruner':        'None (5-fold complete eval)',
    'direction':     'minimize',
    'objective':     'oof_rmse',
    'reference':     axes.REFERENCE,
    'axes':          {k: list(map(str, v)) for k, v in axes.AXES.items()},
    'agg_preset_lib': axes.AGG_PRESET_LIB,
    'pp_pin': {
        'cleaning': axes.PP_PIN_CLEANING,
        'outlier':  axes.PP_PIN_OUTLIER,
        'binarize': axes.PP_PIN_BINARIZE,
        'iso':      axes.PP_PIN_ISO,
        'lds':      axes.PP_PIN_LDS,
        'ge':       axes.PP_PIN_GE,
    },
    'exclude_cols':  axes.EXCLUDE_COLS,
}
with open(META_JSON, 'w') as f:
    json.dump(meta, f, indent=2, default=str, ensure_ascii=False)

print(f'Study : baseline_group')
print(f'Storage: {DB_PATH}')
print(f'기존 trial: {len(study.trials)}')
print(f'meta.json saved → {META_JSON}')

## 3. Objective + study.optimize

axes.AXES dict의 11개 categorical을 trial.suggest_categorical로 샘플 → axes.run_one(cfg, seed=42) 호출.

In [ ]:
def objective(trial):
    cfg = {
        axis: trial.suggest_categorical(axis, options)
        for axis, options in axes.AXES.items()
    }
    result = axes.run_one(
        cfg, seed=42, n_jobs=N_JOBS, n_estimators=N_ESTIMATORS,
    )
    # OOF RMSE를 objective로 (val/test는 attrs 기록만)
    trial.set_user_attr('val_rmse',  result['val_rmse'])
    trial.set_user_attr('test_rmse', result['test_rmse'])
    trial.set_user_attr('elapsed_sec', result['elapsed_sec'])
    trial.set_user_attr('effective_target_transform', result['effective_target_transform'])  # tweedie 자동 OFF 추적
    return result['oof_rmse']

# Optuna trial 직렬 (모델 내부 N_JOBS와 곱셈 효과 방지) — strategy_common §8
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, n_jobs=1, show_progress_bar=True)

## 4. 산출물 저장

- `trials.csv` — study.trials_dataframe()
- `param_importance.csv` — fANOVA 기반 축 중요도 (OAT tornado와 비교용)

optuna.db는 storage로 자동 저장됨.

In [ ]:
import pandas as pd
from optuna.importance import get_param_importances, FanovaImportanceEvaluator

trials_df = study.trials_dataframe()
trials_df.to_csv(os.path.join(OUT_DIR, 'trials.csv'), index=False)

imp = get_param_importances(study, evaluator=FanovaImportanceEvaluator(seed=42))
imp_df = pd.DataFrame([{'axis': k, 'importance': v} for k, v in imp.items()])
imp_df.to_csv(os.path.join(OUT_DIR, 'param_importance.csv'), index=False)

print(f'trials.csv         : {len(trials_df)} rows → {OUT_DIR}/trials.csv')
print(f'param_importance   : {len(imp_df)} axes  → {OUT_DIR}/param_importance.csv')